In [ ]:
# ============================================================
# WEB SCRAPING - BOOKS TO SCRAPE, POR CATEGORIA
# Semana 10 - DML - Adaptado de descargardatos.ipynb
#
# Diferencia clave con descargardatos.ipynb: en vez de recorrer
# las paginas 1 a 50 del catalogo plano (que mezcla todas las
# categorias y no permite saber a cual pertenece cada libro),
# aqui se recorre cada CATEGORIA por separado. Eso nos da el
# campo 'categoria' que la guia de laboratorio necesita para:
#   - la auditoria SELECT DISTINCT categoria
#   - el reto autonomo (seccion 11: la categoria 'Fantasy')
#
# 'Fantasy' se deja FUERA de la lista de EN CLASE a proposito:
# es la que se usa en el reto autonomo (seccion 11 de la guia).
# ============================================================

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

try:
    from google.colab import files
    EN_COLAB = True
except ImportError:
    EN_COLAB = False


# ============================================================
# 1. CONFIGURACION
# ============================================================

BASE = "https://books.toscrape.com/"
CATALOGUE = BASE + "catalogue/"

rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

# Categorias seleccionadas para EN CLASE (scraping responsable:
# pocas categorias, pocas paginas por categoria)
CATEGORIAS_CLASE = {
    "Travel": "catalogue/category/books/travel_2/index.html",
    "Mystery": "catalogue/category/books/mystery_3/index.html",
    "Historical Fiction": "catalogue/category/books/historical-fiction_4/index.html",
}

MAX_PAGINAS_POR_CATEGORIA = 2


# ============================================================
# 2. FUNCION: SCRAPEAR UNA CATEGORIA
# ============================================================

def scrape_categoria(nombre_categoria, url_relativa, max_paginas=MAX_PAGINAS_POR_CATEGORIA):

    datos = []
    url = BASE + url_relativa
    pagina = 1

    while url and pagina <= max_paginas:

        print(f"Scrapeando '{nombre_categoria}' pagina {pagina}...")

        respuesta = requests.get(url, timeout=20)

        if respuesta.status_code != 200:
            print(f"Error en '{nombre_categoria}' pagina {pagina}: HTTP {respuesta.status_code}")
            break

        soup = BeautifulSoup(respuesta.text, "html.parser")

        libros = soup.find_all("article", class_="product_pod")

        for libro in libros:

            nombre = libro.h3.a["title"]

            precio_texto = libro.find("p", class_="price_color").get_text(strip=True)
            precio = float(re.sub(r"[^\d.]", "", precio_texto))

            disponibilidad = libro.find("p", class_="instock availability").get_text(strip=True)

            rating_texto = libro.find("p", class_="star-rating")["class"][1]
            calificacion = rating_map.get(rating_texto)

            enlace = libro.h3.a["href"]
            url_libro = CATALOGUE + enlace.replace("../../../", "").replace("../../", "")

            datos.append({
                "nombre": nombre,
                "precio": precio,
                "categoria": nombre_categoria,
                "disponibilidad": disponibilidad,
                "calificacion": calificacion,
                "url": url_libro
            })

        siguiente = soup.find("li", class_="next")

        if siguiente:
            carpeta_categoria = url.split("/books/")[1].rsplit("/", 1)[0]
            url = BASE + "catalogue/category/books/" + carpeta_categoria + "/" + siguiente.a["href"]
            pagina += 1
        else:
            url = None

        time.sleep(0.2)

    return datos


# ============================================================
# 3. SCRAPEAR TODAS LAS CATEGORIAS DE CLASE
# ============================================================

datos = []

for nombre_cat, ruta_cat in CATEGORIAS_CLASE.items():
    datos += scrape_categoria(nombre_cat, ruta_cat)


# ============================================================
# 4. CREAR DATAFRAME
# ============================================================

df = pd.DataFrame(datos)


# ============================================================
# 5. MOSTRAR RESULTADOS
# ============================================================

print("\n====================================")
print("SCRAPING FINALIZADO")
print("====================================")

print("Total de libros obtenidos:", len(df))
print("\nLibros por categoria:")
print(df["categoria"].value_counts())

print("\nPrimeros registros:")
display(df.head(10))


# ============================================================
# 6. REVISAR LOS DATOS (igual que descargardatos.ipynb)
# ============================================================

print("\nValores nulos:")
print(df.isnull().sum())

print("\nDuplicados:")
print(df.duplicated().sum())


# ============================================================
# 7. EXPORTAR A CSV
# Nota: el nombre debe ser EXACTAMENTE 'productos_web.csv' porque
# es el archivo que espera 02_create_staging.sql / la guia.
# ============================================================

nombre_archivo = "productos_web.csv"

df.to_csv(
    nombre_archivo,
    index=False,
    encoding="utf-8-sig"
)

print("\nArchivo generado:", nombre_archivo)


# ============================================================
# 8. DESCARGAR (solo si estamos en Colab)
# ============================================================

if EN_COLAB:
    files.download(nombre_archivo)
else:
    print("No estamos en Colab: el archivo quedo guardado en el entorno local.")

In [ ]:
# ============================================================
# RETO AUTONOMO (seccion 11 de la guia): categoria Fantasy
#
# No ejecutes esta celda en clase. Es la que cada estudiante usa
# DESPUES, de forma individual, cambiando el nombre del archivo
# para no mezclarlo con productos_web.csv.
# Si no hay Internet, usa el respaldo E_challenge_fantasia_raw.csv
# en vez de ejecutar esta celda (Paso 1 del reto).
# ============================================================

datos_fantasy = scrape_categoria("Fantasy", "catalogue/category/books/fantasy_19/index.html")

df_fantasy = pd.DataFrame(datos_fantasy)

print("Libros de Fantasy obtenidos:", len(df_fantasy))
display(df_fantasy.head(10))

df_fantasy.to_csv("productos_challenge.csv", index=False, encoding="utf-8-sig")
print("\nArchivo generado: productos_challenge.csv")

if EN_COLAB:
    files.download("productos_challenge.csv")